In [ ]:
import json
import random

random.seed(42)

dataset_with_answers_path = "/content/drive/MyDrive/Adaptive_RAG/musique_ans_v1.0_dev.jsonl"
examples_with_answers = []

all_lines = []
with open(dataset_with_answers_path, 'r', encoding='utf-8') as f:
    all_lines = [json.loads(line) for line in f]


examples_with_answers = random.sample(all_lines, 700)[300:400]

print(f"Загружено {len(examples_with_answers)} случайных примеров")

Загружено 100 случайных примеров


In [ ]:
len(examples_with_answers)

100

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
from typing import List, Dict, Any
import re

def transform_dataset(dataset: List[Dict[str, Any]]) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Преобразует датасет в два DataFrame:
    1. paragraphs_df - параграфы с абсолютной нумерацией
    2. questions_df - подвопросы с абсолютными ID параграфов и ID общего вопроса

    В тексте вопросов заменяет #1, #2, #3 и т.д. на ответы соответствующих подвопросов.
    """
    paragraphs_list = []
    questions_list = []
    global_paragraph_counter = 1
    paragraph_mapping = {}

    for example in dataset:
        example_id = example['id']

        for paragraph in example['paragraphs']:
            paragraph_idx = paragraph['idx']
            key = (example_id, paragraph_idx)


            paragraph_mapping[key] = global_paragraph_counter


            paragraphs_list.append({
                'absolute_id': global_paragraph_counter,
                'text': paragraph['paragraph_text']
            })

            global_paragraph_counter += 1


    for example in dataset:
        example_id = example['id']
        main_question_id = example['id']
        main_question_text = example['question']

        decomposition = example.get('question_decomposition', [])


        processed_questions = []

        for i, sub_question in enumerate(decomposition):
            question_text = sub_question['question']


            answer_mapping = {}
            for j in range(i):
                if j < len(processed_questions):
                    placeholder = f"#{j+1}"
                    answer_mapping[placeholder] = decomposition[j].get('answer', '')


            for placeholder, answer in answer_mapping.items():
                if answer:

                    #print(question_text)
                    pattern = re.escape(placeholder)
                    #print(pattern)

                    question_text = re.sub(placeholder, answer, question_text)
                    #print(question_text)

            processed_questions.append(question_text)


        for i, sub_question in enumerate(decomposition):
            paragraph_support_idx = sub_question.get('paragraph_support_idx')


            paragraph_absolute_id = None
            if paragraph_support_idx is not None:
                key = (example_id, paragraph_support_idx)
                paragraph_absolute_id = paragraph_mapping.get(key)

            questions_list.append({
                'question': processed_questions[i],
                'paragraph_absolute_id': paragraph_absolute_id,
                'main_question_id': main_question_id,
                'main_question_text': main_question_text
            })


    paragraphs_df = pd.DataFrame(paragraphs_list)
    questions_df = pd.DataFrame(questions_list)

    return paragraphs_df, questions_df

In [ ]:
paragraphs_df, questions_df = transform_dataset(examples_with_answers)
paragraphs_df

,absolute_id,text
0,1,Liam Thomas Garrigan (born 17 October 1981) is...
1,2,Ideas for a Conan film were proposed as early ...
2,3,"Jeffrey Shawn Swords (born December 27, 1973 i..."
3,4,``Born in the U.S.A. ''is a 1984 song written ...
4,5,"Haji Sahib of Turangzai, the most famous Pukht..."
...,...,...
1993,1994,"In the earlier seasons of Family Guy, Clevelan..."
1994,1995,Lacey Chabert voiced Meg for the first product...
1995,1996,Meg Griffin Family Guy character First appeara...
1996,1997,John Herbert Family Guy character First appear...


In [ ]:
questions_df

,question,paragraph_absolute_id,main_question_id,main_question_text
0,who took the sword out of the stone,11,2hop__81825_49084,Who plays the character that took the sword ou...
1,who plays Arthur in once upon a time,1,2hop__81825_49084,Who plays the character that took the sword ou...
2,Which state is KZAR located?,29,2hop__131455_11960,What is the tallest building in the state wher...
3,What is the tallest building in Texas ?,40,2hop__131455_11960,What is the tallest building in the state wher...
4,MacGruder and Loud >> creator,58,3hop1__694534_160088_85460,What is the average salary of a working person...
...,...,...,...,...
241,When did Kosovo first attend the Olympics game...,1949,4hop1__146971_698949_157828_162309,When did the country that has the same co-offi...
242,Ernie Watts >> place of birth,1973,2hop__274110_413723,In which district was Ernie Watts born?
243,Woolhampton >> located in the administrative t...,1977,2hop__274110_413723,In which district was Ernie Watts born?
244,who does mila kunis play on family guy,1982,2hop__64175_90973,Who did the original voice of the character Mi...


In [ ]:
paragraphs_df = paragraphs_df.drop_duplicates(['text'])
paragraphs_df

,absolute_id,text
0,1,Liam Thomas Garrigan (born 17 October 1981) is...
1,2,Ideas for a Conan film were proposed as early ...
2,3,"Jeffrey Shawn Swords (born December 27, 1973 i..."
3,4,``Born in the U.S.A. ''is a 1984 song written ...
4,5,"Haji Sahib of Turangzai, the most famous Pukht..."
...,...,...
1993,1994,"In the earlier seasons of Family Guy, Clevelan..."
1994,1995,Lacey Chabert voiced Meg for the first product...
1995,1996,Meg Griffin Family Guy character First appeara...
1996,1997,John Herbert Family Guy character First appear...


In [ ]:
paragraphs_df

,absolute_id,text
0,1,Liam Thomas Garrigan (born 17 October 1981) is...
1,2,Ideas for a Conan film were proposed as early ...
2,3,"Jeffrey Shawn Swords (born December 27, 1973 i..."
3,4,``Born in the U.S.A. ''is a 1984 song written ...
4,5,"Haji Sahib of Turangzai, the most famous Pukht..."
...,...,...
1993,1994,"In the earlier seasons of Family Guy, Clevelan..."
1994,1995,Lacey Chabert voiced Meg for the first product...
1995,1996,Meg Griffin Family Guy character First appeara...
1996,1997,John Herbert Family Guy character First appear...


In [ ]:
counter = 0
for r in questions_df.paragraph_absolute_id:
  if r not in (paragraphs_df.absolute_id):
    counter+=1
counter

15

In [ ]:
import random
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

class Retriever:
    def __init__(self, qa_df, paragraphs_df, top_k=5, p_correct=0.5, seed=None,
                 model_name='all-MiniLM-L6-v2', device='cuda'):
        if seed is not None:
            random.seed(seed)
            torch.manual_seed(seed)

        self.top_k = top_k
        self.p_correct = p_correct
        self.device = device

        self.encoder = SentenceTransformer(model_name, device=device)

        self.paragraph_ids = paragraphs_df['absolute_id'].tolist()
        self.paragraph_texts = paragraphs_df['text'].tolist()
        print("Вычисление эмбеддингов параграфов...")
        self.paragraph_embeddings = self.encoder.encode(
            self.paragraph_texts, convert_to_tensor=True, device=device, show_progress_bar=True,batch_size=16
        )
        self.paragraph_embeddings = torch.nn.functional.normalize(self.paragraph_embeddings, p=2, dim=1)

        self.correct_map = {}
        for _, row in qa_df.iterrows():
            q = row['question']
            pid = row['paragraph_absolute_id']
            self.correct_map.setdefault(q, set()).add(pid)

        self.text_by_id = dict(zip(self.paragraph_ids, self.paragraph_texts))

    def get_top_k_relevant(self, query, k=5, page=0):
        query_emb = self.encoder.encode(query, convert_to_tensor=True, device=self.device)
        query_emb = torch.nn.functional.normalize(query_emb, p=2, dim=0)
        similarities = torch.matmul(self.paragraph_embeddings, query_emb)
        sorted_indices = torch.argsort(similarities, descending=True).cpu().numpy()
        sorted_ids = [self.paragraph_ids[i] for i in sorted_indices]
        start = page * k
        end = start + k
        top_ids = sorted_ids[start:end]
        top_texts = [self.text_by_id[pid] for pid in top_ids]
        docs = [f"{pid}. {txt}" for pid, txt in zip(top_ids, top_texts)]

        return {
            'docs': docs,
            'ids': top_ids
        }

    def __call__(self, query, last):
        query_emb = self.encoder.encode(query, convert_to_tensor=True, device=self.device)
        query_emb = torch.nn.functional.normalize(query_emb, p=2, dim=0)
        similarities = torch.matmul(self.paragraph_embeddings, query_emb)
        sorted_indices = torch.argsort(similarities, descending=True).cpu().numpy()
        sorted_ids = [self.paragraph_ids[i] for i in sorted_indices]

        correct_ids = self.correct_map.get(query, set())
        candidates = sorted_ids[:self.top_k]

        if last == False:
            include_correct = random.random() < self.p_correct
        else:
            include_correct = True

        correct_in_candidates = [pid for pid in candidates if pid in correct_ids]

        if include_correct:
            if not correct_in_candidates and correct_ids:
                chosen_correct = random.choice(list(correct_ids))
                replace_idx = random.randint(0, len(candidates) - 1)
                candidates[replace_idx] = chosen_correct
        else:
            if correct_in_candidates:
                candidates_without_correct = [pid for pid in candidates if pid not in correct_ids]
                next_ids = []
                for pid in sorted_ids[self.top_k:]:
                    if pid not in correct_ids and pid not in candidates_without_correct:
                        next_ids.append(pid)
                    if len(candidates_without_correct) + len(next_ids) >= self.top_k:
                        break
                candidates = (candidates_without_correct + next_ids)[:self.top_k]

        random.shuffle(candidates)

        docs = [f"{pid}. {self.text_by_id[pid]}" for pid in candidates]
        ids = [pid for pid in candidates]

        return {
            'docs': docs,
            'ids': ids
        }

In [ ]:
# retriever = BM25Retriever(
#     paragraphs_df=paragraphs_df,
#     questions_df=questions_df
# )
TOP_K = 5
P_CORRECT = 0.3 #0.3
SEED = 42
RETRIEVER_MODEL = 'Qwen/Qwen3-Embedding-4B'
DEVICE = "cuda"

retriever = Retriever(questions_df, paragraphs_df, top_k=TOP_K, p_correct=P_CORRECT,
                             seed=SEED, model_name=RETRIEVER_MODEL, device=DEVICE)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Вычисление эмбеддингов параграфов...


Batches:   0%|          | 0/115 [00:00<?, ?it/s]

In [ ]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "INDEX_SEARCH_TOOL",
            "description": "Retrieve documents from the index by a search query.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Search query"}
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "submit_answer",
            "description": "Submit the final answer with the document ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {"type": "integer", "description": "Document ID that best answers the question"}
                },
                "required": ["id"]
            }
        }
    }
]

In [ ]:
import json
import re
import requests
from typing import List, Dict, Any, Optional, Tuple

class QwenAgent:
    def __init__(self,model_name, api_key: str, base_url: str = "https://openrouter.ai/api/v1"):
        """
        Инициализация агента для OpenRouter API.
        """
        self.model = model_name
        self.tools = TOOLS
        self.api_key = api_key
        self.base_url = base_url
        self.headers = {
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json"
        }

    def _call_model(
        self,
        messages: List[Dict[str, Any]],
        max_tokens: int = 1000,
        max_retries: int = 5
    ) -> Dict[str, Any]:
        """
        Вызывает модель через OpenRouter API.
        Возвращает объект message целиком (не только content).
        """
        payload = {
            "model": self.model,
            "messages": messages,
            "max_tokens": max_tokens,
            "temperature": 0,
            "top_p": 1,
            "stream": False
        }
        if self.tools:
            payload["tools"] = self.tools
        for attempt in range(max_retries):
            response = requests.post(
                f"{self.base_url}/chat/completions",
                headers=self.headers,
                json=payload,
                timeout=60
            )
            if not response.ok:
                print("Ошибка API:", response.status_code)
                print("Детали:", response.json())
            response.raise_for_status()
            result = response.json()
            print()
            print('result', result["choices"][0]["message"])
            print('_'*25)

            return result["choices"][0]["message"]
        raise Exception("Превышено максимальное количество попыток")
    def _extract_tool_call(self, message: Dict[str, Any]) -> Optional[Dict[str, Any]]:
        """
        Извлекает вызов инструмента из объекта message.
        OpenRouter возвращает tool_calls в структурированном виде — парсить текст не нужно.
        """
        tool_calls = message.get("tool_calls")
        if tool_calls and len(tool_calls) > 0:
            tool_call = tool_calls[0]
            arguments = tool_call["function"]["arguments"]
            return {
                "id": tool_call["id"],
                "name": tool_call["function"]["name"],
                "arguments": (
                    json.loads(arguments)
                    if isinstance(arguments, str)
                    else arguments
                )
            }
        # ✅ Способ 2: модель написала вызов инструмента в тексте
        content = message.get("content") or ""
        # Формат: <tool_call>{"name": "...", "arguments": {...}}</tool_call>
        match = re.search(r'<tool_call>\s*(.*?)\s*</tool_call>', content, re.DOTALL)
        if match:
            try:
                data = json.loads(match.group(1))
                return {
                    "id": f"manual_{data.get('name', 'unknown')}",  # синтетический id
                    "name": data.get("name"),
                    "arguments": data.get("arguments", data.get("parameters", {}))
                }
            except json.JSONDecodeError:
                pass

    def step(self, messages: List[Dict[str, Any]]) -> Tuple[Optional[Dict[str, Any]], str]:
        """
        Выполняет один шаг: вызывает модель и извлекает вызов инструмента.

        Args:
            messages: история сообщений

        Returns:
            (tool_call, raw_response)
            tool_call: словарь с name и arguments, если найден, иначе None
            raw_response: полный текст ответа модели
        """
        message = self._call_model(messages)
        tool_call = self._extract_tool_call(message)
        print(tool_call)
        return tool_call, message

In [ ]:
import re
import torch
import torch.nn.functional as F

# ======================== ПРОМПТ АГЕНТА ========================
SYSTEM_PROMPT = """
ROLE:
You are a precise knowledge assistant. Answer the user's question using the provided documents.

RULES:
1. If documents contain the answer, select the most explicit and complete document and call `submit_answer` with its id.
2. If no document clearly answers the question, call `INDEX_SEARCH_TOOL` to retrieve more relevant documents. You may reformulate the query.
3. Do not call `INDEX_SEARCH_TOOL` repeatedly with semantically identical queries.
4. Never invent document ids or content. Only rely on tool-provided observations.

FORMAT:
- Before each tool call, you may provide a short thought (1‑2 sentences, <40 words) inside `<thought>` tags.
- Your final response must be a tool call in the following format:
  `<tool_call>{"name": "<tool_name>", "arguments": {...}}</tool_call>`
- Do not output `<observation>` or document text; the system will provide them after your tool call.

EXAMPLES:

EXAMPLE 1 (answer not in initial documents):
question: Who is the current president of France?
Documents:
31. The French President is elected for a five-year term.
222. The Élysée Palace is the official residence of the President.
413. France has a semi-presidential system.
<thought>These documents give background but don't name the current president. I need a more specific query.</thought>
<tool_call>{"name": "INDEX_SEARCH_TOOL", "arguments": {"query": "Who is the current president of France"}}</tool_call>
<observation> (system provides)
6. Emmanuel Macron was re-elected in 2022.
79. Emmanuel Macron is the current president of France.
</observation>
<thought>Document 79 explicitly names Emmanuel Macron as current president.</thought>
<tool_call>{"name": "submit_answer", "arguments": {"id": 79}}</tool_call>

EXAMPLE 2 (answer found after tool call):
question: When did the Berlin Wall fall?
Documents:
45. The Berlin Wall divided Berlin from 1961 to 1989.
12. The fall paved the way for German reunification.
<thought>The exact date is missing. I'll search for it.</thought>
<tool_call>{"name": "INDEX_SEARCH_TOOL", "arguments": {"query": "Berlin Wall fall date"}}</tool_call>
<observation>
23. The Berlin Wall fell on November 9, 1989.
34. November 9, 1989 is a historic date.
</observation>
<thought>Document 23 gives the exact date.</thought>
<tool_call>{"name": "submit_answer", "arguments": {"id": 23}}</tool_call>

Now begin."""

In [ ]:
OPENROUTER_API_KEY =

llm = QwenAgent(
    api_key = OPENROUTER_API_KEY,
    model_name="openai/gpt-5.4-nano")

In [ ]:
def index_search_tool(query, last, page):
    result = retriever.get_top_k_relevant(query, k=5, page = page) #retriever(query, last)   # result['docs'] — список строк документов

    tool_content = "\n".join(result['docs'])
    return result['ids'], tool_content

In [ ]:
THINK_AGAIN_PROMPT = (
    "Before calling INDEX_SEARCH_TOOL, re‑examine ALL provided documents. "
    "If the answer is present (even indirectly) or can be derived from them, "
    "DO NOT call the tool – answer immediately from the documents. "
    "Only call INDEX_SEARCH_TOOL if you are absolutely certain the required information is missing. "
    "Never guess or invent an answer. When uncertain, always prefer calling the tool over guessing."
)


In [ ]:
def run_agent_for_subquestion(agent, subquestion, initial_docs, ids, correct_ids, max_calls=5, max_steps=5, entropy_threshold=0.0):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"question: {subquestion}\n\nDocuments:\n{initial_docs}"}
    ]
    tool_calls_used = 0
    step_info_history = []
    final_answer_id = None
    success = False
    current_ids = list(ids)

    for step in range(max_steps):
        tool_call, response = agent.step(messages)

        print('tool_call', tool_call)

        if not tool_call:
            print("No valid tool call found. Stopping.")
            break

        name = tool_call.get("name")
        args = tool_call.get("arguments", {})
        tool_call_id = tool_call.get("id")

        has_correct = len(set(current_ids) & set(correct_ids)) > 0

        step_info = {
            "step": step,
            "action": f"{name}({json.dumps(args)})",
            "has_correct": has_correct,
            "text": response,
        }
        step_info_history.append(step_info)
        print("RES:", response.get("content"))

        assistant_message = {
            "role": "assistant",
            "content": response.get("content"),
            "tool_calls": [{
                "id": tool_call_id,
                "type": "function",
                "function": {
                    "name": name,
                    "arguments": json.dumps(args)
                }
            }]
        }
        messages.append(assistant_message)

        if name == "submit_answer":
            final_answer_id = args.get("id")
            success = True
            break

        elif name == "INDEX_SEARCH_TOOL":
            if tool_calls_used >= max_calls:
                print("Max tool calls exceeded, stopping.")
                break

            query = args.get("query", "")
            last = (step == max_steps - 1)
            current_ids, tool_content = index_search_tool(query, last, tool_calls_used+1 )
            #print(type(tool_content))
            has_correct = len(set(current_ids) & set(correct_ids)) > 0

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call_id,
                "name": name,
                "content": tool_content
            })
            tool_calls_used += 1
            continue

        else:
            print(f"Unknown tool: {name}")
            break

    if final_answer_id is None:
        final_answer_id = 'No answer'

    return final_answer_id, step_info_history, success

In [ ]:
def make_serializable(obj):
    if isinstance(obj, set):
        return list(obj)
    if hasattr(obj, 'tolist'):   # для тензоров PyTorch / TensorFlow / NumPy
        return obj.tolist()
    if isinstance(obj, dict):
        return {k: make_serializable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [make_serializable(item) for item in obj]
    return obj

In [ ]:
from tqdm import tqdm
import numpy as np
import json

unique_main_ids = questions_df['main_question_id']  # .unique()
results_by_main = {}

MAX_TOOL_CALLS = 5
#ENTROPY_THRESHOLD = 0.07590062288278732
SAVE_PATH = '/content/drive/MyDrive/Adaptive_RAG/gpt_5_4_nano_adaptive_qwen.json' #deepseek/deepseek-v3.2



for main_id in tqdm(unique_main_ids, desc="Main questions"):
    #print(f"\n{'='*50}")
    #print(f"Обработка основного вопроса ID: {main_id}")

    subquestions_df = questions_df[questions_df['main_question_id'] == main_id]
    subquestions = subquestions_df['question'].tolist()

    results_for_main = {}
    all_sub_correct = True

    for subq in tqdm(subquestions, desc=f"Подвопросы main_{main_id}", leave=False):
        #print(f"\n  Подвопрос: {subq}")

        first_page = retriever.get_top_k_relevant(subq, k=5, page = 0)#retriever(subq, False)
        ids = first_page['ids']
        initial_docs_str = "\n".join(first_page['docs'])
        correct_ids = set(subquestions_df[subquestions_df['question'] == subq]['paragraph_absolute_id'])

        has_correct = len(set(ids) & correct_ids) > 0
        #print("  Начальные документы (есть правильный? {}):".format(has_correct))
        #print(initial_docs_str)


        doc_id, logits_hist, success = run_agent_for_subquestion(
            llm,
            subq,
            initial_docs_str,
            ids,
            correct_ids,
            max_calls=MAX_TOOL_CALLS,
            entropy_threshold=-1
        )
        try:
            doc_id = int(doc_id)
        except:
            doc_id = doc_id

        is_correct = doc_id in correct_ids
        if not is_correct:
            all_sub_correct = False

        results_for_main[subq] = {
            "found_doc": doc_id,
            "correct_ids": list(correct_ids),
            "is_correct": is_correct,
            "logits_history": logits_hist,
            "initial_has_correct": has_correct,
            "success": success
        }


        #print(f"  Агент вернул: {doc_id} (правильные: {correct_ids}) -> {'✓' if is_correct else '✗'}")

    results_by_main[main_id] = {
        "subquestions": results_for_main,
        "main_correct": all_sub_correct,
        "total_subquestions": len(subquestions),
        "correct_subquestions": sum(1 for r in results_for_main.values() if r['is_correct'])
    }

    correct_count = results_by_main[main_id]["correct_subquestions"]
    total = results_by_main[main_id]["total_subquestions"]
    #print(f"\n  Итоги по основному вопросу {main_id}: правильно {correct_count}/{total} ({correct_count/total*100:.1f}%)")
    #print(f"  Основной вопрос в целом {'✓ верно' if all_sub_correct else '✗ неверно'}")

    serializable_results = make_serializable(results_by_main)
    with open(SAVE_PATH, 'w', encoding='utf-8') as f:
        json.dump(serializable_results, f, ensure_ascii=False, indent=2)
    #print(f"  Промежуточные результаты сохранены: {SAVE_PATH}")

In [ ]:
1+1

In [ ]:
import json
with open(SAVE_PATH, 'r', encoding='utf-8') as f:
    daresults_by_mainta = json.load(f)

In [ ]:
import numpy as np

def compute_metrics(results_by_main):
    total_subq = 0
    correct_subq = 0
    success_subq = 0
    steps_per_subq = []
    initial_has_correct_stats = {'total': 0, 'correct': 0}
    initial_no_correct_stats = {'total': 0, 'correct': 0}

    main_stats = {}

    for main_id, main_data in results_by_main.items():
        # Проверяем, что main_data - словарь и содержит 'subquestions'
        if not isinstance(main_data, dict):
            print(f"Предупреждение: для main_id {main_id} данные не словарь ({type(main_data)}), пропускаем.")
            continue

        subqs = main_data.get('subquestions', {})
        if not isinstance(subqs, dict):
            print(f"Предупреждение: для main_id {main_id} subquestions не словарь ({type(subqs)}), пропускаем.")
            continue

        num_subq = len(subqs)
        num_correct_main = 0

        for subq, res in subqs.items():
            if not isinstance(res, dict):
                print(f"Предупреждение: для подвопроса {subq} данные не словарь ({type(res)}), пропускаем.")
                continue

            total_subq += 1

            # Определяем правильность ответа
            is_correct = res.get('is_correct', False)
            if is_correct:
                correct_subq += 1
                num_correct_main += 1

            # Успешность завершения
            if res.get('success', False):
                success_subq += 1

            # Количество шагов
            steps = len(res.get('logits_history', []))
            steps_per_subq.append(steps)

            # Статистика по начальной выдаче
            initial_has_correct = res.get('initial_has_correct', False)
            if initial_has_correct:
                initial_has_correct_stats['total'] += 1
                if is_correct:
                    initial_has_correct_stats['correct'] += 1
            else:
                initial_no_correct_stats['total'] += 1
                if is_correct:
                    initial_no_correct_stats['correct'] += 1

        main_stats[main_id] = {
            'num_subq': num_subq,
            'num_correct': num_correct_main,
            'all_correct': num_correct_main == num_subq,
            'main_correct': main_data.get('main_correct', False)  # можно добавить для сравнения
        }

    total_main = len(main_stats)
    main_all_correct = sum(1 for v in main_stats.values() if v['all_correct'])

    metrics = {
        'total_subquestions': total_subq,
        'subq_accuracy': correct_subq / total_subq if total_subq else 0,
        'subq_success_rate': success_subq / total_subq if total_subq else 0,
        'avg_steps_per_subq': np.mean(steps_per_subq) if steps_per_subq else 0,
        'std_steps_per_subq': np.std(steps_per_subq) if steps_per_subq else 0,
        'min_steps_per_subq': min(steps_per_subq) if steps_per_subq else 0,
        'max_steps_per_subq': max(steps_per_subq) if steps_per_subq else 0,
        'total_main_questions': total_main,
        'main_accuracy': main_all_correct / total_main if total_main else 0,
        'main_all_correct_count': main_all_correct,
        'subq_accuracy_when_initial_has_correct': (
            initial_has_correct_stats['correct'] / initial_has_correct_stats['total']
            if initial_has_correct_stats['total'] else 0
        ),
        'subq_accuracy_when_initial_no_correct': (
            initial_no_correct_stats['correct'] / initial_no_correct_stats['total']
            if initial_no_correct_stats['total'] else 0
        ),
        'main_stats': main_stats
    }

    return metrics

def print_metrics(metrics):
    """Красивый вывод метрик."""
    print("\n" + "="*60)
    print("ИТОГОВЫЕ МЕТРИКИ")
    print("="*60)
    print(f"Всего подвопросов: {metrics['total_subquestions']}")
    print(f"Точность на подвопросах (subq_accuracy): {metrics['subq_accuracy']:.2%}")
    print(f"Доля успешных завершений агента: {metrics['subq_success_rate']:.2%}")
    print(f"Среднее количество шагов на подвопрос: {metrics['avg_steps_per_subq']:.2f} ± {metrics['std_steps_per_subq']:.2f} "
          f"(мин: {metrics['min_steps_per_subq']}, макс: {metrics['max_steps_per_subq']})")
    print(f"\nВсего основных вопросов: {metrics['total_main_questions']}")
    print(f"Основные вопросы, где все подвопросы отвечены верно: {metrics['main_all_correct_count']} "
          f"({metrics['main_accuracy']:.2%})")
    print(f"\nТочность на подвопросах:")
    print(f"  - когда правильный документ был в начальной выдаче: {metrics['subq_accuracy_when_initial_has_correct']:.2%}")
    print(f"  - когда правильного документа не было в начальной выдаче: {metrics['subq_accuracy_when_initial_no_correct']:.2%}")

    print("\nДетализация по основным вопросам:")
    for main_id, stats in metrics['main_stats'].items():
        print(f"  Main {main_id}: {stats['num_correct']}/{stats['num_subq']} верных подвопросов "
              f"({'все верны' if stats['all_correct'] else 'не все'}) (main_correct={stats.get('main_correct', 'N/A')})")

In [ ]:
metrics = compute_metrics(daresults_by_mainta)
print_metrics(metrics)

In [ ]:
SAVE_PATH = '/content/drive/MyDrive/Adaptive_RAG/results_gpt_5_4_nano.json'

In [ ]:
1+1